# Simulación Fase de Grupos - Mundial 2026

Simulación de la fase de grupos del Mundial 2026 usando distribución Poisson basada en ratings Elo

In [29]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import random
from collections import defaultdict, Counter

## Cargar ratings

In [30]:
# Cargar ratings
ratings_df = pd.read_csv('output/ratings_championship.csv')
ratings_dict = dict(zip(ratings_df['Equipo'], ratings_df['Rating']))
print(f"Ratings cargados: {len(ratings_dict)} equipos")

Ratings cargados: 282 equipos


## Funciones de simulación

In [31]:
def get_team_rating(team_name, ratings_df):
    """Obtener el rating de un equipo"""
    rating = ratings_df[ratings_df['Equipo'] == team_name]['Rating']
    if len(rating) == 0:
        raise ValueError(f"Equipo no encontrado: {team_name}")
    return rating.values[0]

def calculate_lambda(home_team, away_team, ratings_df):
    """Calcular lambda (goles esperados) para cada equipo"""
    home_rating = get_team_rating(home_team, ratings_df)
    away_rating = get_team_rating(away_team, ratings_df)
    rating_diff = home_rating - away_rating

    lambda_home = 1.35 + (rating_diff / 200)
    lambda_away = 1.35 - (rating_diff / 300)

    lambda_home = max(0.2, lambda_home)
    lambda_away = max(0.2, lambda_away)

    return lambda_home, lambda_away

def simulate_match(home_team, away_team, ratings_df, seed=None):
    """Simular un partido y devolver el resultado"""
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    lambda_home, lambda_away = calculate_lambda(home_team, away_team, ratings_df)

    # Simular goles usando Poisson
    goals_home = np.random.poisson(lambda_home)
    goals_away = np.random.poisson(lambda_away)

    return goals_home, goals_away

def simulate_group(teams, ratings_df, group_name, seed=None):
    """Simular un grupo de 4 equipos - devuelve tabla y partidos"""
    if seed is not None:
        np.random.seed(seed)

    # Inicializar tabla
    table = defaultdict(lambda: {'pts': 0, 'gf': 0, 'ga': 0, 'gd': 0, 'w': 0, 'd': 0, 'l': 0})

    # Lista de partidos
    matches = []

    # Todos contra todos
    for i, home in enumerate(teams):
        for j, away in enumerate(teams):
            if i >= j:
                continue

            gh, ga = simulate_match(home, away, ratings_df)

            # Guardar partido
            matches.append({
                'group': group_name,
                'home': home,
                'away': away,
                'home_goals': gh,
                'away_goals': ga,
                'score': f"{gh}-{ga}"
            })

            # Actualizar estadísticas
            table[home]['gf'] += gh
            table[home]['ga'] += ga
            table[away]['gf'] += ga
            table[away]['ga'] += gh

            if gh > ga:
                table[home]['pts'] += 3
                table[home]['w'] += 1
                table[away]['l'] += 1
            elif gh < ga:
                table[away]['pts'] += 3
                table[away]['w'] += 1
                table[home]['l'] += 1
            else:
                table[home]['pts'] += 1
                table[away]['pts'] += 1
                table[home]['d'] += 1
                table[away]['d'] += 1

    # Calcular diferencia de gol
    for team in table:
        table[team]['gd'] = table[team]['gf'] - table[team]['ga']

    # Crear DataFrame ordenado
    results = []
    for team in teams:
        results.append({
            'group': group_name,
            'team': team,
            'pts': table[team]['pts'],
            'w': table[team]['w'],
            'd': table[team]['d'],
            'l': table[team]['l'],
            'gf': table[team]['gf'],
            'ga': table[team]['ga'],
            'gd': table[team]['gd']
        })

    df = pd.DataFrame(results)
    df = df.sort_values(['pts', 'gd', 'gf'], ascending=[False, False, False])
    df['position'] = range(1, 5)

    # DataFrame de partidos
    matches_df = pd.DataFrame(matches)

    return df, matches_df

print("Funciones definidas correctamente")

Funciones definidas correctamente


## Definir grupos del Mundial 2026

In [32]:
## Definir grupos personalizados del Mundial 2026

# Equipos personalizados por grupo (en inglés)
custom_groups = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['United States', 'Paraguay', 'Australia', 'Turkey'],
    'E': ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador'],
    'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama']
}

# Usar grupos personalizados
groups = custom_groups

# Mostrar grupos con ratings
print("=== GRUPOS MUNDIAL 2026 (PERSONALIZADOS) ===\n")
for name, teams in groups.items():
    print(f"Grupo {name}:")
    for team in teams:
        rating = ratings_dict.get(team, 'N/A')
        if rating != 'N/A':
            print(f"  - {team} ({rating:.0f})")
        else:
            print(f"  - {team} (NO ENCONTRADO)")
    print()

=== GRUPOS MUNDIAL 2026 (PERSONALIZADOS) ===

Grupo A:
  - Mexico (1571)
  - South Africa (1559)
  - South Korea (1663)
  - Czech Republic (1553)

Grupo B:
  - Canada (1541)
  - Bosnia and Herzegovina (1487)
  - Qatar (1540)
  - Switzerland (1641)

Grupo C:
  - Brazil (1663)
  - Morocco (1678)
  - Haiti (1543)
  - Scotland (1578)

Grupo D:
  - United States (1553)
  - Paraguay (1518)
  - Australia (1653)
  - Turkey (1654)

Grupo E:
  - Germany (1680)
  - Curaçao (1574)
  - Ivory Coast (1620)
  - Ecuador (1600)

Grupo F:
  - Netherlands (1708)
  - Japan (1704)
  - Sweden (1551)
  - Tunisia (1640)

Grupo G:
  - Belgium (1659)
  - Egypt (1586)
  - Iran (1659)
  - New Zealand (1578)

Grupo H:
  - Spain (1808)
  - Cape Verde (1585)
  - Saudi Arabia (1583)
  - Uruguay (1603)

Grupo I:
  - France (1776)
  - Senegal (1637)
  - Iraq (1611)
  - Norway (1646)

Grupo J:
  - Argentina (1753)
  - Algeria (1631)
  - Austria (1611)
  - Jordan (1563)

Grupo K:
  - Portugal (1692)
  - DR Congo (1583)
  

## Simulación única de fase de grupos

In [47]:
# Simular todos los grupos una vez
np.random.seed(42)
all_group_results = []
all_matches_results = []

for group_name, teams in groups.items():
    df, matches_df = simulate_group(teams, ratings_df, group_name)
    all_group_results.append(df)
    all_matches_results.append(matches_df)

# Concatenar resultados
final_table = pd.concat(all_group_results, ignore_index=True)
matches_table = pd.concat(all_matches_results, ignore_index=True)

# Mostrar tabla completa
print("=== RESULTADO FASE DE GRUPOS (UNA SIMULACIÓN) ===\n")
for group_name in group_names:
    group_df = final_table[final_table['group'] == group_name][['position', 'team', 'pts', 'w', 'd', 'l', 'gf', 'ga', 'gd']]
    print(f"\n--- Grupo {group_name} ---")
    print(group_df.to_string(index=False))

=== RESULTADO FASE DE GRUPOS (UNA SIMULACIÓN) ===


--- Grupo A ---
 position           team  pts  w  d  l  gf  ga  gd
        1         Mexico    7  2  1  0   6   2   4
        2    South Korea    5  1  2  0   1   0   1
        3 Czech Republic    2  0  2  1   3   4  -1
        4   South Africa    1  0  1  2   1   5  -4

--- Grupo B ---
 position                   team  pts  w  d  l  gf  ga  gd
        1            Switzerland    7  2  1  0   6   1   5
        2 Bosnia and Herzegovina    5  1  2  0   2   1   1
        3                 Canada    4  1  1  1   3   6  -3
        4                  Qatar    0  0  0  3   2   5  -3

--- Grupo C ---
 position     team  pts  w  d  l  gf  ga  gd
        1  Morocco    9  3  0  0  10   2   8
        2   Brazil    4  1  1  1   3   7  -4
        3    Haiti    3  1  0  2   5   4   1
        4 Scotland    1  0  1  2   2   7  -5

--- Grupo D ---
 position          team  pts  w  d  l  gf  ga  gd
        1     Australia    6  2  0  1   5   3   2
      

In [50]:
all_matches_results

[  group          home            away  home_goals  away_goals score
 0     A        Mexico    South Africa           3           0   3-0
 1     A        Mexico     South Korea           0           0   0-0
 2     A        Mexico  Czech Republic           3           2   3-2
 3     A  South Africa     South Korea           0           1   0-1
 4     A  South Africa  Czech Republic           1           1   1-1
 5     A   South Korea  Czech Republic           0           0   0-0,
   group                    home                    away  home_goals  \
 0     B                  Canada  Bosnia and Herzegovina           1   
 1     B                  Canada                   Qatar           2   
 2     B                  Canada             Switzerland           0   
 3     B  Bosnia and Herzegovina                   Qatar           1   
 4     B  Bosnia and Herzegovina             Switzerland           0   
 5     B                   Qatar             Switzerland           1   
 
    away_g

## Monte Carlo - Probabilidades de clasificación

In [53]:
def run_monte_carlo(groups, ratings_df, n_sims=1000):
    """Ejecutar múltiples simulaciones de fase de grupos"""

    # Contadores
    position_counts = defaultdict(lambda: defaultdict(int))

    print(f"Iniciando Monte Carlo con {n_sims} simulaciones...")
    print("=" * 50)

    # Mostrar grupos una vez
    for group_name, teams in groups.items():
        print(f"Grupo {group_name}: {', '.join(teams)}")
    print("=" * 50)

    for sim in range(n_sims):
        if sim % 1000 == 0:
            print(f"Simulación {sim}/{n_sims}...")

        for group_name, teams in groups.items():
            df, matches_df = simulate_group(teams, ratings_df, group_name, seed=sim)

            for _, row in df.iterrows():
                team = row['team']
                pos = row['position']
                position_counts[team][pos] += 1

    print(f"Simulación {n_sims}/{n_sims}... Completado!")
    print("=" * 50)

    # Calcular probabilidades
    probs = []
    for team, counts in position_counts.items():
        row = {'team': team}
        for pos in range(1, 5):
            row[f'pos_{pos}'] = counts[pos] / n_sims * 100
        # Probabilidad de clasificar (top 2)
        row['classified'] = (counts[1] + counts[2]) / n_sims * 100
        # Probabilidad de mejor 3ro
        row['best_third'] = counts[3] / n_sims * 100
        probs.append(row)

    return pd.DataFrame(probs).sort_values('pos_1', ascending=False)

# Ejecutar Monte Carlo
print("Ejecutando Monte Carlo (10,000 simulaciones)...")
n_sims = 100
prob_df = run_monte_carlo(groups, ratings_df, n_sims=n_sims)
print(f"Completado!")

# Mostrar probabilidades
print("\n=== PROBABILIDADES DE CLASIFICACIÓN ===")
print("\nTop 20 - Probabilidad de quedar 1ro:")
display(prob_df.head(20)[['team', 'pos_1', 'pos_2', 'classified', 'best_third']].round(1))

Ejecutando Monte Carlo (10,000 simulaciones)...
Iniciando Monte Carlo con 100 simulaciones...
Grupo A: Mexico, South Africa, South Korea, Czech Republic
Grupo B: Canada, Bosnia and Herzegovina, Qatar, Switzerland
Grupo C: Brazil, Morocco, Haiti, Scotland
Grupo D: United States, Paraguay, Australia, Turkey
Grupo E: Germany, Curaçao, Ivory Coast, Ecuador
Grupo F: Netherlands, Japan, Sweden, Tunisia
Grupo G: Belgium, Egypt, Iran, New Zealand
Grupo H: Spain, Cape Verde, Saudi Arabia, Uruguay
Grupo I: France, Senegal, Iraq, Norway
Grupo J: Argentina, Algeria, Austria, Jordan
Grupo K: Portugal, DR Congo, Uzbekistan, Colombia
Grupo L: England, Croatia, Ghana, Panama
Simulación 0/100...
Simulación 100/100... Completado!
Completado!

=== PROBABILIDADES DE CLASIFICACIÓN ===

Top 20 - Probabilidad de quedar 1ro:


,team,pos_1,pos_2,classified,best_third
28,Spain,83.0,11.0,94.0,6.0
32,France,68.0,21.0,89.0,6.0
36,Argentina,65.0,19.0,84.0,12.0
6,Switzerland,64.0,19.0,83.0,12.0
44,England,63.0,24.0,87.0,8.0
0,South Korea,54.0,18.0,72.0,23.0
40,Portugal,51.0,22.0,73.0,15.0
11,Morocco,47.0,36.0,83.0,8.0
12,Australia,46.0,33.0,79.0,15.0
16,Germany,45.0,24.0,69.0,20.0


## Visualizaciones

In [54]:
# Gráfico de barras - Probabilidad de ser 1ro
fig = px.bar(
    prob_df.head(20),
    x='team',
    y='pos_1',
    title='Probabilidad de quedar 1ro en el grupo (Top 20)',
    labels={'pos_1': 'Probabilidad (%)', 'team': 'Equipo'},
    color='pos_1',
    color_continuous_scale='Blues'
)
fig.update_layout(xaxis={'categoryorder': 'total descending'})
fig.show()

In [10]:
# Gráfico de barras agrupadas - Probabilidades por posición
top_teams = prob_df.head(16)['team'].tolist()
plot_df = prob_df[prob_df['team'].isin(top_teams)]

fig = go.Figure()
for pos in [1, 2, 3, 4]:
    fig.add_trace(go.Bar(
        name=f'Posición {pos}',
        x=plot_df['team'],
        y=plot_df[f'pos_{pos}'],
    ))

fig.update_layout(
    barmode='group',
    title='Probabilidad por posición (Top 16 equipos)',
    xaxis_title='Equipo',
    yaxis_title='Probabilidad (%)',
    template='plotly_white'
)
fig.show()

In [11]:
# Heatmap de probabilidades
heatmap_df = prob_df[prob_df['team'].isin(top_teams)][['team', 'pos_1', 'pos_2', 'pos_3', 'pos_4']].copy()
heatmap_df = heatmap_df.set_index('team')
heatmap_df.columns = ['1ro', '2do', '3ro', '4to']

fig = px.imshow(
    heatmap_df,
    text_auto='.1f',
    aspect='auto',
    title='Mapa de probabilidades de posición (%)',
    color_continuous_scale='RdYlGn_r'
)
fig.show()

## Tabla de clasificación completa

In [12]:
# Mostrar tabla completa de probabilidades
print("=== PROBABILIDADES COMPLETAS DE CLASIFICACIÓN ===\n")
display(prob_df.round(1))

=== PROBABILIDADES COMPLETAS DE CLASIFICACIÓN ===



,team,pos_1,pos_2,pos_3,pos_4,classified,best_third
0,Spain,36.0,25.0,26.0,13.0,61.0,26.0
7,Japan,35.0,26.0,21.0,18.0,61.0,21.0
15,Belgium,34.0,25.0,21.0,20.0,59.0,21.0
43,Curaçao,33.0,26.0,21.0,20.0,59.0,21.0
31,Uruguay,33.0,24.0,23.0,20.0,57.0,23.0
27,Denmark,33.0,25.0,21.0,21.0,58.0,21.0
23,Algeria,33.0,25.0,26.0,16.0,58.0,26.0
39,Ghana,32.0,25.0,21.0,22.0,57.0,21.0
35,Saudi Arabia,32.0,26.0,21.0,21.0,58.0,21.0
47,Greece,32.0,26.0,21.0,21.0,58.0,21.0


In [ ]:
# Guardar resultados
prob_df.to_csv('output/group_stage_probabilities.csv', index=False)
print("Resultados guardados en output/group_stage_probabilities.csv")